In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
import time, json

spark = (SparkSession.builder
    .appName("UrgenceNet-Partie5-Pipeline")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print(spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 23:28:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


3.5.3


In [2]:
event_schema = StructType([
    StructField("visit_id", StringType()), StructField("patient_id", StringType()),
    StructField("facility_id", StringType()), StructField("city", StringType()),
    StructField("status", StringType()), StructField("priority", IntegerType()),
    StructField("wait_minutes", IntegerType()), StructField("department", StringType()),
    StructField("desk_id", StringType()), StructField("transfer_to", StringType()),
    StructField("event_time", TimestampType()),
])
facility_schema = StructType([
    StructField("facility_id", StringType()), StructField("name", StringType()),
    StructField("city", StringType()), StructField("hospital_group", StringType()),
    StructField("type", StringType()), StructField("beds", IntegerType()),
    StructField("latitude", DoubleType()), StructField("longitude", DoubleType()),
])

fac = (spark.read.schema(facility_schema).json("s3a://lake/bronze/reference/facilities.json")
       .select("facility_id", F.col("name").alias("facility_name"),
               F.col("type").alias("facility_type"), "hospital_group"))

raw = (spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "urgences")
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load())

events = (raw.select(F.from_json(F.col("value").cast("string"), event_schema).alias("e"))
             .select("e.*"))

# Dédoublonnage + watermark. event_time est ajouté à la clé : les doublons du générateur ont le même
# event_time, donc le résultat est identique à (visit_id, status), et Spark peut purger l'état.
clean = (events
    .where("visit_id IS NOT NULL AND status IS NOT NULL AND event_time IS NOT NULL")
    .withWatermark("event_time", "10 minutes")
    .dropDuplicates(["visit_id", "status", "event_time"]))

# Jointure stream-static + wait_band (sans UDF)
enriched = (clean.join(F.broadcast(fac), "facility_id", "left")
    .withColumn("wait_band",
        F.when(F.col("wait_minutes") < 15, "court")
         .when(F.col("wait_minutes") <= 45, "moyen")
         .when(F.col("wait_minutes") > 45, "long"))
    .withColumn("date", F.to_date("event_time")))
enriched.printSchema()

26/11/05 23:29:25 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


root
 |-- facility_id: string (nullable = true)
 |-- visit_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- status: string (nullable = true)
 |-- priority: integer (nullable = true)
 |-- wait_minutes: integer (nullable = true)
 |-- department: string (nullable = true)
 |-- desk_id: string (nullable = true)
 |-- transfer_to: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- facility_name: string (nullable = true)
 |-- facility_type: string (nullable = true)
 |-- hospital_group: string (nullable = true)
 |-- wait_band: string (nullable = true)
 |-- date: date (nullable = true)



In [17]:
SILVER = "s3a://lake/silver/urgences_clean"
GOLD   = "s3a://lake/gold/kpi_facility_10min"

def start_silver():
    return (enriched.writeStream.queryName("silver_clean")
        .format("parquet").outputMode("append")
        .partitionBy("city", "date")
        .option("path", SILVER)
        .option("checkpointLocation", "s3a://lake/checkpoints/silver_urgences_clean")
        .trigger(processingTime="30 seconds").start())

# Fenêtre tumbling 10 min : passages (ARRIVED) et délai moyen (CARE_STARTED)
kpi = (enriched
    .groupBy(F.window("event_time", "10 minutes").alias("w"), "facility_id", "city", "hospital_group")
    .agg(F.count(F.when(F.col("status") == "ARRIVED", 1)).alias("arrivals"),
         F.round(F.avg(F.when(F.col("status") == "CARE_STARTED", F.col("wait_minutes"))), 2).alias("avg_wait_min"))
    .select(F.col("w.start").alias("window_start"), F.col("w.end").alias("window_end"),
            "facility_id", "city", "hospital_group", "arrivals", "avg_wait_min"))

def start_gold():
    return (kpi.writeStream.queryName("gold_kpi")
        .format("delta").outputMode("append")
        .option("path", GOLD)
        .option("checkpointLocation", "s3a://lake/checkpoints/gold_kpi_10min")
        .trigger(processingTime="30 seconds").start())

q_silver = start_silver()
q_gold = start_gold()

26/11/06 15:10:25 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/06 15:10:26 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/06 15:10:35 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/11/06 15:10:45 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/11/06 15:11:27 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 42242 milliseconds
26/11/06 15:15:31 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 31785 milliseconds
26/11/06 15:

In [4]:
for q in spark.streams.active:
    p = q.lastProgress
    print(q.name, "|", q.status["message"], "| batch", p["batchId"] if p else None,
          "| lignes", p["numInputRows"] if p else None)

silver_clean | Processing new data | batch None | lignes None
gold_kpi | Initializing sources | batch None | lignes None


In [5]:
s = spark.read.parquet(SILVER)
print("Silver :", s.count(), "lignes | sans doublon (visit_id, status) :",
      s.dropDuplicates(["visit_id", "status"]).count())
s.groupBy("city", "date").count().orderBy("city", "date").show()

g = spark.read.format("delta").load(GOLD)
print("Gold :", g.count(), "lignes")
g.orderBy(F.desc("window_start")).show(8, truncate=False)

spark.sql(f"DESCRIBE HISTORY delta.`{GOLD}`") \
     .select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

26/11/05 23:30:04 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
[Stage 1:>                  (0 + 1) / 1][Stage 2:>                  (0 + 1) / 1]

AnalysisException: Unable to infer schema for Parquet at . It must be specified manually.

In [ ]:
from minio import Minio
mc = Minio("minio:9000", "minioadmin", "minioadmin123", secure=False)

def last_offsets(ck, k=1):
    names = [o.object_name for o in mc.list_objects("lake", prefix=f"checkpoints/{ck}/offsets/")
             if o.object_name.rsplit("/", 1)[1].isdigit()]
    names.sort(key=lambda n: int(n.rsplit("/", 1)[1]))
    for n in names[-k:]:
        print("--", n)
        print(mc.get_object("lake", n).read().decode())

26/11/05 23:32:29 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 158318 milliseconds
26/11/06 14:12:42 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 52812734 milliseconds
26/11/06 14:13:20 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 52996530 milliseconds
26/11/06 14:13:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [6]:
for q in spark.streams.active:
    p = q.lastProgress
    print(q.name, "|", q.status["message"], "| batch", p["batchId"] if p else None)

silver_clean | Processing new data | batch 12
gold_kpi | Processing new data | batch 3


26/11/06 14:22:40 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 85444 milliseconds
26/11/06 14:23:33 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 33024 milliseconds
26/11/06 14:23:46 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 65907 milliseconds
26/11/06 14:24:18 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 32318 milliseconds
26/11/06 14:24:50 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 32279 milliseconds
26/11/06 14:25:13 WARN FileStreamSinkLog: Compacting took 2295 ms for compact batch 19
26/11/06 14:26:03 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 43601 millisec

In [7]:
s = spark.read.parquet(SILVER)
print("Silver :", s.count(), "lignes")
s.groupBy("city", "date").count().orderBy("city", "date").show()

26/11/06 14:34:16 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 42279 milliseconds
                                                                                

Silver : 25156 lignes


+----------+----------+-----+
|      city|      date|count|
+----------+----------+-----+
|Casablanca|2026-11-05|  266|
|Casablanca|2026-11-06| 5500|
| Marrakech|2026-11-05|  293|
| Marrakech|2026-11-06| 6718|
|     Rabat|2026-11-05|  263|
|     Rabat|2026-11-06| 5946|
|    Tanger|2026-11-05|  306|
|    Tanger|2026-11-06| 5864|
+----------+----------+-----+



26/11/06 14:35:32 WARN FileStreamSinkLog: Compacting took 2875 ms for compact batch 39
26/11/06 14:35:33 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 33632 milliseconds
26/11/06 14:35:38 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 81603 milliseconds
26/11/06 14:36:22 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 44377 milliseconds
                                                                                

In [8]:
total = s.count()
distincts = s.dropDuplicates(["visit_id", "status"]).count()
print("Lignes :", total, "| distinctes (visit_id, status) :", distincts, "| doublons restants :", total - distincts)

26/11/06 14:37:24 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 54036 milliseconds
[Stage 385:============================>                            (2 + 2) / 4]

Lignes : 25156 | distinctes (visit_id, status) : 25156 | doublons restants : 0


In [9]:
g = spark.read.format("delta").load(GOLD)
print("Gold :", g.count(), "lignes")
g.orderBy(F.desc("window_start")).show(8, truncate=False)

26/11/06 14:38:00 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 36522 milliseconds
26/11/06 14:38:17 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 114524 milliseconds
                                                                                

Gold : 1918 lignes


[Stage 413:>                (0 + 2) / 4][Stage 414:========>        (2 + 2) / 4]

+-------------------+-------------------+-----------+----------+--------------+--------+------------+
|window_start       |window_end         |facility_id|city      |hospital_group|arrivals|avg_wait_min|
+-------------------+-------------------+-----------+----------+--------------+--------+------------+
|2026-11-06 20:30:00|2026-11-06 20:40:00|F-036      |Tanger    |GHT-Nord      |8       |34.0        |
|2026-11-06 20:30:00|2026-11-06 20:40:00|F-017      |Rabat     |GHT-Atlantique|0       |21.0        |
|2026-11-06 20:30:00|2026-11-06 20:40:00|F-031      |Tanger    |GHT-Nord      |24      |58.63       |
|2026-11-06 20:30:00|2026-11-06 20:40:00|F-012      |Rabat     |GHT-Atlantique|0       |24.6        |
|2026-11-06 20:30:00|2026-11-06 20:40:00|F-019      |Rabat     |GHT-Atlantique|4       |5.0         |
|2026-11-06 20:30:00|2026-11-06 20:40:00|F-007      |Casablanca|GHT-Atlantique|0       |NULL        |
|2026-11-06 20:30:00|2026-11-06 20:40:00|F-027      |Marrakech |GHT-Sud       |7  

26/11/06 14:38:59 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 42700 milliseconds
26/11/06 14:40:16 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 77126 milliseconds
26/11/06 14:41:21 WARN FileStreamSinkLog: Compacting took 7038 ms for compact batch 49
26/11/06 14:41:23 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 53669 milliseconds
26/11/06 14:41:43 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 86595 milliseconds
26/11/06 14:42:32 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 49372 milliseconds
                                                                                

In [10]:
g = spark.read.format("delta").load(GOLD)
print("Lignes :", g.count(),
      "| clés distinctes (facility_id, window_start) :",
      g.select("facility_id", "window_start").distinct().count())

spark.sql(f"DESCRIBE HISTORY delta.`{GOLD}`") \
     .select("version", "timestamp", "operation", "operationMetrics") \
     .show(10, truncate=False)

26/11/06 14:43:49 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 49523 milliseconds
26/11/06 14:44:59 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 146580 milliseconds
                                                                                

Lignes : 2391 | clés distinctes (facility_id, window_start) : 2510


26/11/06 14:45:53 WARN FileStreamSinkLog: Compacting took 3181 ms for compact batch 59
                                                                                

+-------+-------------------+----------------+-----------------------------------------------------------------------------------------+
|version|timestamp          |operation       |operationMetrics                                                                         |
+-------+-------------------+----------------+-----------------------------------------------------------------------------------------+
|30     |2026-11-06 14:43:55|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 119, numOutputBytes -> 10471, numAddedFiles -> 4}|
|29     |2026-11-06 14:42:24|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 78, numOutputBytes -> 9895, numAddedFiles -> 4}  |
|28     |2026-11-06 14:41:33|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 78, numOutputBytes -> 9895, numAddedFiles -> 4}  |
|27     |2026-11-06 14:40:02|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 159, numOutputBytes -> 10873, numAddedFiles -> 4}|
|26     |2026-11-06 14:38:56|STREAMING UP

26/11/06 14:45:56 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 56881 milliseconds
26/11/06 14:46:46 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 50625 milliseconds
26/11/06 14:47:24 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 37261 milliseconds
                                                                                

In [11]:
from delta.tables import DeltaTable

v = DeltaTable.forPath(spark, GOLD).history(1).select("version").first()[0]
g = spark.read.format("delta").option("versionAsOf", v).load(GOLD)

n = g.count()
d = g.select("facility_id", "window_start").distinct().count()
print("Version figée :", v)
print("Lignes :", n, "| clés distinctes :", d, "| doublons :", n - d)

26/11/06 14:48:17 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 53765 milliseconds
26/11/06 14:49:03 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 45096 milliseconds
                                                                                

Version figée : 33
Lignes : 2866 | clés distinctes : 2866 | doublons : 0


26/11/06 14:49:50 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 47600 milliseconds
26/11/06 14:50:43 WARN FileStreamSinkLog: Compacting took 3445 ms for compact batch 69
26/11/06 14:50:43 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 30746 milliseconds
26/11/06 14:52:10 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 57869 milliseconds
26/11/06 14:52:51 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 41163 milliseconds
26/11/06 14:55:44 WARN FileStreamSinkLog: Compacting took 4730 ms for compact batch 79
26/11/06 14:56:00 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 30589 milliseconds


In [12]:
q_silver.stop()
q_gold.stop()
print("silver actif :", q_silver.isActive, "| gold actif :", q_gold.isActive)

26/11/06 14:56:08 ERROR FileFormatWriter: Aborting job f1147be1-9b8f-4242-974a-a717c60a8147.
java.lang.InterruptedException
	at java.base/java.util.concurrent.FutureTask.awaitDone(FutureTask.java:418)
	at java.base/java.util.concurrent.FutureTask.get(FutureTask.java:203)
	at org.apache.spark.sql.execution.exchange.BroadcastExchangeExec.doExecuteBroadcast(BroadcastExchangeExec.scala:212)
	at org.apache.spark.sql.execution.InputAdapter.doExecuteBroadcast(WholeStageCodegenExec.scala:517)
	at org.apache.spark.sql.execution.SparkPlan.$anonfun$executeBroadcast$1(SparkPlan.scala:208)
	at org.apache.spark.sql.execution.SparkPlan.$anonfun$executeQuery$1(SparkPlan.scala:246)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:151)
	at org.apache.spark.sql.execution.SparkPlan.executeQuery(SparkPlan.scala:243)
	at org.apache.spark.sql.execution.SparkPlan.executeBroadcast(SparkPlan.scala:204)
	at org.apache.spark.sql.execution.joins.BroadcastHashJoinExec.prepareBroadcast(B

silver actif : False | gold actif : False


In [13]:
s_av = spark.read.parquet(SILVER)
n_s = s_av.count()
d_s = s_av.dropDuplicates(["visit_id", "status"]).count()

v_av = DeltaTable.forPath(spark, GOLD).history(1).select("version").first()[0]
g_av = spark.read.format("delta").option("versionAsOf", v_av).load(GOLD)
n_g = g_av.count()
d_g = g_av.select("facility_id", "window_start").distinct().count()

print("AVANT arrêt -> Silver :", n_s, "lignes,", n_s - d_s, "doublons")
print("AVANT arrêt -> Gold   : version", v_av, "|", n_g, "lignes,", n_g - d_g, "doublons")

[Stage 800:===============================================>         (5 + 1) / 6]

AVANT arrêt -> Silver : 52325 lignes, 0 doublons
AVANT arrêt -> Gold   : version 48 | 3574 lignes, 0 doublons


In [14]:
k = (spark.read.format("kafka")
     .option("kafka.bootstrap.servers", "kafka:9092")
     .option("subscribe", "urgences")
     .option("startingOffsets", "earliest")
     .option("endingOffsets", "latest")
     .load())
k.groupBy("partition").agg((F.max("offset") + 1).alias("end_offset")) \
 .orderBy("partition").show()
print("Total messages dans Kafka :", k.count())

26/11/06 15:01:31 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/11/06 15:01:38 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+---------+----------+
|partition|end_offset|
+---------+----------+
|        0|     20303|
|        1|     20146|
|        2|     20481|
+---------+----------+



[Stage 809:======================================>                  (2 + 1) / 3]

Total messages dans Kafka : 61070


In [15]:
base = "s3a://lake/checkpoints/"
Path = spark._jvm.org.apache.hadoop.fs.Path
fs = Path(base).getFileSystem(spark._jsc.hadoopConfiguration())
it = fs.listFiles(Path(base), True)
while it.hasNext():
    p = it.next().getPath().toString()
    if ("/offsets/" in p or "/commits/" in p) and not p.endswith(".tmp"):
        print(p)

s3a://lake/checkpoints/bronze_urgences_raw/commits/0
s3a://lake/checkpoints/bronze_urgences_raw/commits/1
s3a://lake/checkpoints/bronze_urgences_raw/commits/2
s3a://lake/checkpoints/bronze_urgences_raw/commits/3
s3a://lake/checkpoints/bronze_urgences_raw/commits/4
s3a://lake/checkpoints/bronze_urgences_raw/commits/5
s3a://lake/checkpoints/bronze_urgences_raw/commits/6
s3a://lake/checkpoints/bronze_urgences_raw/commits/7
s3a://lake/checkpoints/bronze_urgences_raw/offsets/0
s3a://lake/checkpoints/bronze_urgences_raw/offsets/1
s3a://lake/checkpoints/bronze_urgences_raw/offsets/2
s3a://lake/checkpoints/bronze_urgences_raw/offsets/3
s3a://lake/checkpoints/bronze_urgences_raw/offsets/4
s3a://lake/checkpoints/bronze_urgences_raw/offsets/5
s3a://lake/checkpoints/bronze_urgences_raw/offsets/6
s3a://lake/checkpoints/bronze_urgences_raw/offsets/7
s3a://lake/checkpoints/bronze_urgences_raw/offsets/8
s3a://lake/checkpoints/gold_kpi_10min/commits/0
s3a://lake/checkpoints/gold_kpi_10min/commits/1
s3a

In [16]:
for name, n in [("silver_urgences_clean", 79), ("silver_urgences_clean", 80),
                ("gold_kpi_10min", 48), ("gold_kpi_10min", 49)]:
    print("=====", name, "| offsets/%d" % n)
    for r in spark.read.text(f"s3a://lake/checkpoints/{name}/offsets/{n}").collect():
        print(r.value)

===== silver_urgences_clean | offsets/79
v1
{"batchWatermarkMs":1794021132000,"batchTimestampMs":1793976930057,"conf":{"spark.sql.streaming.stateStore.providerClass":"org.apache.spark.sql.execution.streaming.state.HDFSBackedStateStoreProvider","spark.sql.streaming.join.stateFormatVersion":"2","spark.sql.streaming.stateStore.compression.codec":"lz4","spark.sql.streaming.stateStore.rocksdb.formatVersion":"5","spark.sql.streaming.statefulOperator.useStrictDistribution":"true","spark.sql.streaming.flatMapGroupsWithState.stateFormatVersion":"2","spark.sql.streaming.multipleWatermarkPolicy":"min","spark.sql.streaming.aggregation.stateFormatVersion":"2","spark.sql.shuffle.partitions":"4"}}
{"urgences":{"2":17850,"1":17793,"0":17732}}
===== silver_urgences_clean | offsets/80
v1
{"batchWatermarkMs":1794021733000,"batchTimestampMs":1793976960064,"conf":{"spark.sql.streaming.stateStore.providerClass":"org.apache.spark.sql.execution.streaming.state.HDFSBackedStateStoreProvider","spark.sql.streamin

In [ ]:
q_silver = start_silver()
q_gold = start_gold()

In [18]:
for name, q in [("silver", q_silver), ("gold", q_gold)]:
    p = q.recentProgress[0]
    src = p["sources"][0]
    print(name, "| batchId :", p["batchId"])
    print("   startOffset :", src["startOffset"])
    print("   endOffset   :", src["endOffset"])
    print("   lignes lues :", p["numInputRows"])

silver | batchId : 80
   startOffset : {'urgences': {'2': 17850, '1': 17793, '0': 17732}}
   endOffset   : {'urgences': {'2': 18091, '1': 17963, '0': 17924}}
   lignes lues : 603
gold | batchId : 49
   startOffset : {'urgences': {'2': 17850, '1': 17793, '0': 17732}}
   endOffset   : {'urgences': {'2': 18091, '1': 17964, '0': 17924}}
   lignes lues : 604


26/11/06 15:19:22 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 52431 milliseconds
[Stage 1050:===========================>                          (25 + 4) / 50]

In [19]:
s_ap = spark.read.parquet(SILVER).cache()
n_s2 = s_ap.count()
d_s2 = s_ap.dropDuplicates(["visit_id", "status"]).count()
s_ap.unpersist()

v_ap = DeltaTable.forPath(spark, GOLD).history(1).select("version").first()[0]
g_ap = spark.read.format("delta").option("versionAsOf", v_ap).load(GOLD)
n_g2 = g_ap.count()
d_g2 = g_ap.select("facility_id", "window_start").distinct().count()

print("APRES redémarrage -> Silver :", n_s2, "lignes,", n_s2 - d_s2, "doublons  (avant : 52325)")
print("APRES redémarrage -> Gold   : version", v_ap, "|", n_g2, "lignes,", n_g2 - d_g2, "doublons  (avant : 3574, version 48)")

26/11/06 15:21:46 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 76352 milliseconds
26/11/06 15:21:49 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 122171 milliseconds
26/11/06 15:22:27 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 37875 milliseconds
                                                                                

APRES redémarrage -> Silver : 82232 lignes, 0 doublons  (avant : 52325)
APRES redémarrage -> Gold   : version 67 | 5465 lignes, 0 doublons  (avant : 3574, version 48)


26/11/06 15:23:04 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 36900 milliseconds
26/11/06 15:24:10 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 66640 milliseconds
26/11/06 15:24:54 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 43545 milliseconds
26/11/06 15:25:14 WARN FileStreamSinkLog: Compacting took 2166 ms for compact batch 109
26/11/06 15:26:04 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 41680 milliseconds
26/11/06 15:27:30 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 30742 milliseconds
26/11/06 15:28:39 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 38779 millise

In [20]:
for name, q in [("silver", q_silver), ("gold", q_gold)]:
    p = q.lastProgress
    src = p["sources"][0]
    print(name, "| batchId :", p["batchId"])
    print("   endOffset    :", src["endOffset"])
    print("   latestOffset :", src.get("latestOffset"))

[Stage 1253:==========>                                           (10 + 4) / 50]

silver | batchId : 117
   endOffset    : {'urgences': {'2': 31903, '1': 31519, '0': 32203}}
   latestOffset : {'urgences': {'2': 31903, '1': 31519, '0': 32203}}
gold | batchId : 79
   endOffset    : {'urgences': {'2': 31772, '1': 31400, '0': 32044}}
   latestOffset : {'urgences': {'2': 31772, '1': 31400, '0': 32044}}


26/11/06 15:30:02 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 32041 milliseconds
26/11/06 15:30:27 WARN FileStreamSinkLog: Compacting took 6956 ms for compact batch 119
26/11/06 15:30:33 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 31677 milliseconds
26/11/06 15:31:10 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 37003 milliseconds
26/11/06 15:31:26 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 141889 milliseconds
[Stage 1289:>               (0 + 4) / 4][Stage 1291:>               (0 + 0) / 4]

In [21]:
print([k for k in dir() if any(w in k.lower() for w in ("slid", "sess", "tumb", "kpi", "q_"))])

['SparkSession', '__session__', 'kpi', 'q_gold', 'q_silver']


26/11/06 15:32:24 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 58621 milliseconds
26/11/06 15:33:15 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 50179 milliseconds
26/11/06 15:34:40 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 40560 milliseconds
26/11/06 15:35:15 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 35027 milliseconds
[Stage 1359:==========>                                           (10 + 4) / 50]

In [22]:
q_silver.stop()
q_gold.stop()

sliding = (enriched.filter(F.col("status") == "TRIAGED")
    .groupBy(F.window("event_time", "10 minutes", "5 minutes").alias("w"),
             "facility_id", "city", "hospital_group")
    .agg(F.count("*").alias("triaged"),
         F.sum(F.when(F.col("priority") <= 2, 1).otherwise(0)).alias("prioritaires"))
    .select(F.col("w.start").alias("window_start"), F.col("w.end").alias("window_end"),
            "facility_id", "city", "hospital_group", "triaged", "prioritaires",
            F.round(F.col("prioritaires") / F.col("triaged"), 3).alias("taux_prioritaires")))

q_slide = (sliding.writeStream.queryName("kpi_sliding")
    .format("memory").outputMode("update")
    .option("checkpointLocation", "s3a://lake/checkpoints/test_sliding")
    .trigger(processingTime="30 seconds").start())
print("sliding actif :", q_slide.isActive)

26/11/06 15:35:27 WARN TaskSetManager: Lost task 12.0 in stage 1359.0 (TID 9337) (172.21.0.5 executor 1): TaskKilled (Stage cancelled: Job 796 cancelled part of cancelled job group c3c87aa7-bd7f-4b74-b7ea-b314de02697d)
26/11/06 15:35:27 WARN TaskSetManager: Lost task 10.0 in stage 1359.0 (TID 9335) (172.21.0.5 executor 1): TaskKilled (Stage cancelled: Job 796 cancelled part of cancelled job group c3c87aa7-bd7f-4b74-b7ea-b314de02697d)
26/11/06 15:35:28 WARN TaskSetManager: Lost task 42.0 in stage 1359.0 (TID 9336) (172.21.0.6 executor 0): TaskKilled (Stage cancelled: Job 796 cancelled part of cancelled job group c3c87aa7-bd7f-4b74-b7ea-b314de02697d)
26/11/06 15:35:30 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


sliding actif : True


26/11/06 15:35:33 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


In [25]:
spark.sql("SELECT * FROM kpi_sliding ORDER BY window_start DESC, taux_prioritaires DESC").show(10, truncate=False)

+-------------------+-------------------+-----------+----------+--------------+-------+------------+-----------------+
|window_start       |window_end         |facility_id|city      |hospital_group|triaged|prioritaires|taux_prioritaires|
+-------------------+-------------------+-----------+----------+--------------+-------+------------+-----------------+
|2026-11-07 16:50:00|2026-11-07 17:00:00|F-031      |Tanger    |GHT-Nord      |1      |1           |1.0              |
|2026-11-07 16:50:00|2026-11-07 17:00:00|F-011      |Rabat     |GHT-Atlantique|1      |1           |1.0              |
|2026-11-07 16:50:00|2026-11-07 17:00:00|F-026      |Marrakech |GHT-Sud       |1      |1           |1.0              |
|2026-11-07 16:50:00|2026-11-07 17:00:00|F-020      |Rabat     |GHT-Atlantique|1      |1           |1.0              |
|2026-11-07 16:50:00|2026-11-07 17:00:00|F-037      |Tanger    |GHT-Nord      |3      |2           |0.667            |
|2026-11-07 16:50:00|2026-11-07 17:00:00|F-029  

In [26]:
spark.sql("""
SELECT window_start, window_end, triaged, prioritaires, taux_prioritaires
FROM kpi_sliding
WHERE facility_id = 'F-021'
ORDER BY window_start DESC
""").show(8, truncate=False)

+-------------------+-------------------+-------+------------+-----------------+
|window_start       |window_end         |triaged|prioritaires|taux_prioritaires|
+-------------------+-------------------+-------+------------+-----------------+
|2026-11-07 17:25:00|2026-11-07 17:35:00|9      |2           |0.222            |
|2026-11-07 17:20:00|2026-11-07 17:30:00|4      |1           |0.25             |
|2026-11-07 17:20:00|2026-11-07 17:30:00|23     |6           |0.261            |
|2026-11-07 17:15:00|2026-11-07 17:25:00|16     |2           |0.125            |
|2026-11-07 17:15:00|2026-11-07 17:25:00|26     |5           |0.192            |
|2026-11-07 17:10:00|2026-11-07 17:20:00|3      |1           |0.333            |
|2026-11-07 17:10:00|2026-11-07 17:20:00|15     |2           |0.133            |
|2026-11-07 17:05:00|2026-11-07 17:15:00|3      |1           |0.333            |
+-------------------+-------------------+-------+------------+-----------------+
only showing top 8 rows



In [27]:
df = spark.table("kpi_sliding").filter("facility_id = 'F-021'")
print("Lignes brutes :", df.count(), "| fenêtres distinctes :", df.select("window_start").distinct().count())

spark.sql("""
SELECT window_start, window_end, city, triaged, prioritaires, taux_prioritaires
FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY facility_id, window_start ORDER BY triaged DESC) AS rn
      FROM kpi_sliding WHERE facility_id = 'F-021')
WHERE rn = 1
ORDER BY window_start DESC
""").show(8, truncate=False)

Lignes brutes : 396 | fenêtres distinctes : 367


+-------------------+-------------------+---------+-------+------------+-----------------+
|window_start       |window_end         |city     |triaged|prioritaires|taux_prioritaires|
+-------------------+-------------------+---------+-------+------------+-----------------+
|2026-11-07 19:20:00|2026-11-07 19:30:00|Marrakech|8      |0           |0.0              |
|2026-11-07 19:15:00|2026-11-07 19:25:00|Marrakech|19     |2           |0.105            |
|2026-11-07 19:10:00|2026-11-07 19:20:00|Marrakech|17     |4           |0.235            |
|2026-11-07 19:05:00|2026-11-07 19:15:00|Marrakech|8      |2           |0.25             |
|2026-11-07 19:00:00|2026-11-07 19:10:00|Marrakech|6      |0           |0.0              |
|2026-11-07 18:55:00|2026-11-07 19:05:00|Marrakech|12     |1           |0.083            |
|2026-11-07 18:50:00|2026-11-07 19:00:00|Marrakech|23     |5           |0.217            |
|2026-11-07 18:45:00|2026-11-07 18:55:00|Marrakech|27     |6           |0.222            |

In [28]:
q_slide.stop()

sessions = (enriched.filter((F.col("status") == "TRIAGED") & F.col("desk_id").isNotNull())
    .groupBy("desk_id", F.session_window("event_time", "5 minutes").alias("s"))
    .agg(F.count("*").alias("nb_triages"))
    .select("desk_id",
            F.col("s.start").alias("session_start"),
            F.col("s.end").alias("session_end"),
            F.round((F.col("s.end").cast("long") - F.col("s.start").cast("long")) / 60 - 5, 1).alias("duree_activite_min"),
            "nb_triages"))

q_session = (sessions.writeStream.queryName("desk_sessions")
    .format("memory").outputMode("update")
    .option("checkpointLocation", "s3a://lake/checkpoints/test_session")
    .trigger(processingTime="30 seconds").start())
print("session active :", q_session.isActive)

26/11/06 15:44:25 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


AnalysisException: Update output mode not supported for session window on streaming DataFrames/DataSets;
Project [desk_id#54, s#59638-T600000ms.start AS session_start#59661, s#59638-T600000ms.end AS session_end#59662, round(((cast((cast(s#59638-T600000ms.end as bigint) - cast(s#59638-T600000ms.start as bigint)) as double) / cast(60 as double)) - cast(5 as double)), 1) AS duree_activite_min#59663, nb_triages#59656L]
+- Aggregate [desk_id#54, session_window#59657-T600000ms], [desk_id#54, session_window#59657-T600000ms AS s#59638-T600000ms, count(1) AS nb_triages#59656L]
   +- Filter isnotnull(event_time#56-T600000ms)
      +- Project [named_struct(start, precisetimestampconversion(precisetimestampconversion(event_time#56-T600000ms, TimestampType, LongType), LongType, TimestampType), end, knownnullable(precisetimestampconversion(precisetimestampconversion(cast(event_time#56-T600000ms + cast(5 minutes as interval) as timestamp), TimestampType, LongType), LongType, TimestampType))) AS sessi

In [29]:
sessions = (enriched.filter((F.col("status") == "TRIAGED") & F.col("desk_id").isNotNull())
    .groupBy("desk_id", F.session_window("event_time", "5 minutes").alias("s"))
    .agg(F.count("*").alias("nb_triages"))
    .select("desk_id",
            F.col("s.start").alias("session_start"),
            F.col("s.end").alias("session_end"),
            F.round((F.col("s.end").cast("long") - F.col("s.start").cast("long")) / 60 - 5, 1).alias("duree_activite_min"),
            "nb_triages"))

q_session = (sessions.writeStream.queryName("desk_sessions")
    .format("memory").outputMode("append")
    .option("checkpointLocation", "s3a://lake/checkpoints/test_session_append")
    .trigger(processingTime="30 seconds").start())
print("session active :", q_session.isActive)

26/11/06 15:47:52 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


session active : True


26/11/06 15:47:55 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


In [31]:
spark.sql("SELECT * FROM desk_sessions ORDER BY session_start DESC").show(10, truncate=False)
print("Sessions fermées :", spark.table("desk_sessions").count())

+--------+-------------------+-------------------+------------------+----------+
|desk_id |session_start      |session_end        |duree_activite_min|nb_triages|
+--------+-------------------+-------------------+------------------+----------+
|T-TAN-11|2026-11-07 21:27:07|2026-11-07 21:32:07|0.0               |1         |
|T-RAB-05|2026-11-07 21:26:55|2026-11-07 21:31:55|0.0               |1         |
|T-RAB-08|2026-11-07 21:26:50|2026-11-07 21:31:50|0.0               |1         |
|T-TAN-12|2026-11-07 21:26:49|2026-11-07 21:31:49|0.0               |1         |
|T-MAR-07|2026-11-07 21:25:08|2026-11-07 21:30:08|0.0               |1         |
|T-CAS-18|2026-11-07 21:24:58|2026-11-07 21:29:58|0.0               |1         |
|T-TAN-17|2026-11-07 21:24:47|2026-11-07 21:29:47|0.0               |1         |
|T-CAS-13|2026-11-07 21:24:31|2026-11-07 21:30:50|1.3               |2         |
|T-TAN-13|2026-11-07 21:22:51|2026-11-07 21:30:57|3.1               |3         |
|T-CAS-02|2026-11-07 21:22:3

[Stage 1474:==========================================>             (3 + 1) / 4]

Sessions fermées : 7370


In [33]:
t = spark.table("desk_sessions")
t.selectExpr("count(*) AS sessions",
             "round(avg(duree_activite_min),2) AS duree_moy_min",
             "max(duree_activite_min) AS duree_max_min",
             "round(avg(nb_triages),2) AS triages_moy",
             "sum(CASE WHEN nb_triages = 1 THEN 1 ELSE 0 END) AS sessions_1_triage").show()

t.orderBy(F.desc("duree_activite_min")).show(8, truncate=False)

q_session.stop()

+--------+-------------+-------------+-----------+-----------------+
|sessions|duree_moy_min|duree_max_min|triages_moy|sessions_1_triage|
+--------+-------------+-------------+-----------+-----------------+
|    7477|         4.39|         48.9|        4.1|             3069|
+--------+-------------+-------------+-----------+-----------------+

+--------+-------------------+-------------------+------------------+----------+
|desk_id |session_start      |session_end        |duree_activite_min|nb_triages|
+--------+-------------------+-------------------+------------------+----------+
|T-MAR-01|2026-11-07 00:40:16|2026-11-07 01:34:07|48.9              |49        |
|T-CAS-02|2026-11-07 16:00:13|2026-11-07 16:54:02|48.8              |32        |
|T-RAB-02|2026-11-07 11:18:01|2026-11-07 11:52:25|29.4              |21        |
|T-MAR-01|2026-11-07 10:07:43|2026-11-07 10:38:58|26.3              |23        |
|T-TAN-02|2026-11-07 14:28:24|2026-11-07 14:59:21|26.0              |26        |
|T-CAS

In [34]:
spark.sql("DESCRIBE HISTORY delta.`s3a://lake/gold/kpi_facility_10min`") \
     .select("version", "timestamp", "operation", "operationMetrics") \
     .orderBy(F.desc("version")).show(6, truncate=60)

+-------+-------------------+----------------+------------------------------------------------------------+
|version|          timestamp|       operation|                                            operationMetrics|
+-------+-------------------+----------------+------------------------------------------------------------+
|     86|2026-11-06 15:35:13|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 40, numOutputByte...|
|     85|2026-11-06 15:34:33|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 39, numOutputByte...|
|     84|2026-11-06 15:33:48|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 39, numOutputByte...|
|     83|2026-11-06 15:33:31|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 78, numOutputByte...|
|     82|2026-11-06 15:33:03|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 197, numOutputByt...|
|     81|2026-11-06 15:32:21|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 40, numOutputByte...|
+-------+-------------------

26/11/06 15:59:12 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/blockmgr-8fb45de3-848a-43f7-9cd4-a882c49bedd4. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/blockmgr-8fb45de3-848a-43f7-9cd4-a882c49bedd4
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:166)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:109)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:90)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1126)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1(DiskBlockManager.scala:368)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1$adapted(DiskBlockManager.scala:364)
	at scala.collection.IndexedSeqOptimize